# Joining Tables and Layers

**Series:** Geospatial Python Basics (notebook 3 of 4)
**Time:** about 25 minutes

**In this notebook you will:**
- Join a table to a layer by a shared ID (attribute join)
- Fix the most common reason joins fail: mismatched ID types
- Join layers by location (spatial join)

**You should already know:** GeoDataFrames and CRS (notebooks 04a and 04b)

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

Data often comes in pieces: shapes in one file, statistics in another. A **join** combines them.

| Join | Matches rows by | Example |
|---|---|---|
| **Attribute join** | A shared ID column | County boundaries + a table of county statistics, by FIPS code |
| **Spatial join** | Location | Which city limits each store falls inside |

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

counties = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/indiana_counties.geojson")
boundaries = counties[["fips", "name", "geometry"]]      # shapes only, as if from a boundary file
stats = pd.read_csv("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/indiana_county_stats.csv")   # numbers only, from another source
boundaries.head(2)

## Attribute join

`merge()` matches rows that have the same value in the key column. First check that the key has the same
type in both tables.

In [ ]:
print(boundaries["fips"].dtype, stats["fips"].dtype)

The boundaries store `fips` as text (shown as `object` or `str`, depending on the pandas version), but pandas read the CSV's `fips` as a number (`int64`).
Text `"18157"` and number `18157` do not match, so a join would fail:

In [ ]:
try:
    boundaries.merge(stats, on="fips")
except ValueError as err:
    print("Join failed:", err)

Fix it by converting the number to text. For codes that can start with zero (states, counties and tracts in
many states), also pad with `zfill()` to the right length: county FIPS codes have 5 characters.

In [ ]:
stats["fips"] = stats["fips"].astype(str).str.zfill(5)
joined = boundaries.merge(stats.drop(columns="name"), on="fips", how="left")
print(len(boundaries), "boundaries,", len(stats), "rows of statistics,", len(joined), "joined")
print(joined["pop_2012"].isna().sum(), "counties without a match")
joined.head(3)

`how="left"` keeps every boundary even if it has no match, so missing matches show up as empty values
instead of silently disappearing. Always check the counts after a join. Keep the GeoDataFrame on the
**left** of `merge()` so the result is still a GeoDataFrame you can map.

In [ ]:
ax = joined.plot(column="unemp_2016", cmap="YlOrRd", legend=True, edgecolor="white", figsize=(5, 7))
ax.set_axis_off()
ax.set_title("Joined statistics, mapped")
plt.show()

## Spatial join

A spatial join matches features by where they are. We ask which of the 64 food stores fall **inside** the
city limits of Lafayette and West Lafayette. Both layers must share a CRS first.

In [ ]:
stores = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/grocery_stores.geojson")
city = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/study_area.geojson")
print(stores.crs == city.crs)
city[["name"]]

In [ ]:
inside = gpd.sjoin(stores, city[["name", "geometry"]], how="left", predicate="within")
inside["in_city"] = inside["index_right"].notna()
print(inside["in_city"].value_counts())
pd.crosstab(inside["store_type"], inside["in_city"])

`predicate="within"` keeps matches where the point lies inside the polygon. Other predicates include
`"intersects"` and `"contains"`. With `how="left"`, stores outside every polygon are kept with empty values,
so you can count them too.

In [ ]:
ax = city.plot(color="whitesmoke", edgecolor="black", figsize=(6, 6))
inside.plot(ax=ax, column="in_city", categorical=True, legend=True, markersize=15,
            legend_kwds={"title": "Inside city limits"})
ax.set_axis_off()
plt.show()

A very common next step is to **count points per polygon**: spatial join the points to the polygons, then
`groupby()` the polygon ID and count.

In [ ]:
in_county = gpd.sjoin(stores, counties[["name", "geometry"]], predicate="within")
in_county.groupby("name").size()

## Try it yourself

1. How many **supermarkets** lie outside the city limits?
2. List the names of the stores outside the city limits.

<details>
<summary><b>Show a solution</b></summary>

```python
outside = inside[~inside["in_city"]]          # ~ means "not"
print((outside["store_type"] == "Supermarket").sum(), "supermarkets outside")
print(outside["store_name"].tolist())
```

</details>

In [ ]:
# Write your answer here

## Recap

- An attribute join (`merge()`) matches rows by a shared ID
- Make ID columns the same type (text, padded with `zfill()`) before joining, and check the counts after
- A spatial join (`gpd.sjoin()`) matches features by location; both layers need the same CRS
- Spatial join plus `groupby()` counts points per polygon

**Next:** [4d Raster Data with rasterio](https://colab.research.google.com/github/SpatialTurn/Foundations/blob/main/notebooks/04d_Raster_Data_with_rasterio.ipynb)